# 01 · Load MedGemma

## Setup: code, libraries, data, login

In [1]:
import os, sys

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # 1. Get the code: your reasoning branch from GitHub
    if not os.path.exists("/content/BN4601A"):
        !git clone -b reasoning https://github.com/likingood16/BN4601A.git /content/BN4601A
    else:
        !git -C /content/BN4601A pull
    %cd /content/BN4601A

    # 2. Libraries Colab doesn't have (Colab already has torch, pandas, numpy, scipy, matplotlib, pillow, tqdm, joblib)
    !pip install -q -U transformers accelerate bitsandbytes   # load MedGemma in 4-bit
    !pip install -q -U peft trl                               # LoRA fine-tuning (task 6, optional)
    !pip install -q -U datasets kagglehub                     # HF datasets (SynthVision, MedQA, MultiCaRe) + KTAS download
    !pip install -q scikit-learn==1.9.1                       # Caleb's saved .joblib models need this exact version

    # 3. KTAS data (not on GitHub): same place as on the laptop
    if not os.path.exists("vitals/data/raw/ktas_triage.csv"):
        import kagglehub, shutil
        os.makedirs("vitals/data/raw", exist_ok=True)
        shutil.copy(kagglehub.dataset_download("ilkeryildiz/emergency-service-triage-application") + "/data.csv",
                    "vitals/data/raw/ktas_triage.csv")

    # 4. Hugging Face login (MedGemma is gated). getpass hides the token and keeps it out of the notebook
    from huggingface_hub import login, whoami
    try:
        whoami()
    except Exception:
        from getpass import getpass
        login(token=getpass("Hugging Face token: "))
else:
    %cd ../..

# Caleb's code (VitalsModel, ktas_data) importable from here on
sys.path.insert(0, "vitals/src")
print("IN_COLAB:", IN_COLAB, "| cwd:", os.getcwd(), "| KTAS data:", os.path.exists("vitals/data/raw/ktas_triage.csv"))

C:\Users\likin\Documents\BN4601A\group project\BN4601A
IN_COLAB: False | cwd: C:\Users\likin\Documents\BN4601A\group project\BN4601A | KTAS data: True


## Device settings

In [2]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0), torch.cuda.is_bf16_supported())
# Colab T4: no native bf16 -> use float16 there; laptop RTX 3050: bfloat16
DTYPE = torch.float16 if IN_COLAB else torch.bfloat16
print("DTYPE:", DTYPE)

True NVIDIA GeForce RTX 3050 4GB Laptop GPU True
DTYPE: torch.bfloat16


## Load model 

In [3]:
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig

MODEL_ID = "google/medgemma-1.5-4b-it"
processor = AutoProcessor.from_pretrained(MODEL_ID)

if IN_COLAB:
    # Colab T4 (15 GB): full size fits
    model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, dtype=DTYPE, device_map="auto")
else:
    # Laptop RTX 3050 (4 GB): only fits in 4-bit
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=DTYPE,
    )
    model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map={"": 0})   # everything on GPU 0; "auto" wrongly offloads to CPU on 4 GB

C:\Users\likin\anaconda3\envs\bn4601a\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1023 19:39:33.167000 27584 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


W1023 19:39:39.041000 27584 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/883 [00:02<31:13,  2.12s/it]

...

Loading weights:  86%|████████▌ | 761/883 [00:11<00:00, 304.49it/s]

Loading weights:  90%|█████████ | 796/883 [00:12<00:00, 317.65it/s]

Loading weights:  95%|█████████▌| 839/883 [00:12<00:00, 341.88it/s]

Loading weights:  99%|█████████▉| 875/883 [00:12<00:00, 343.71it/s]

Loading weights: 100%|██████████| 883/883 [00:12<00:00, 71.73it/s] 

ask()

In [4]:
def ask(system, user):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": system}]},
        {"role": "user",   "content": [{"type": "text", "text": user}]},
    ]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    return processor.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

Test + GPU memory cell

In [5]:
import torch

answer = ask(
    "You are an emergency department doctor. Answer briefly.",
    "A 67-year-old man has had crushing chest pain for 40 minutes. What are the 3 most important red flags to check?",
)
print(answer)

print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

1.  **ECG:** Look for ST-segment elevation, which indicates a possible heart attack.
2.  **Troponin:** Check for elevated levels, indicating heart muscle damage.
3.  **Blood pressure and heart rate:** Assess for signs of shock or instability.
GPU memory used: 3.2 GB
